In [1]:
def load_image_np(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        img = img.convert("RGB")
        return np.array(img)

NameError: name 'Path' is not defined

In [ ]:
def center_crop(image: np.ndarray, crop_size: int = 48) -> np.ndarray:
    h, w, _ = image.shape
    start_y = (h - crop_size) // 2
    start_x = (w - crop_size) // 2
    return image[start_y:start_y+crop_size, start_x:start_x+crop_size]

In [ ]:
def flip_horizontal(image: np.ndarray) -> np.ndarray:
    return image[:, ::-1]

In [ ]:
def normalize_01(image: np.ndarray) -> np.ndarray:
    return image.astype(np.float32) / 255.0

In [ ]:
def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    r = image_float[:, :, 0]
    g = image_float[:, :, 1]
    b = image_float[:, :, 2]
    return 0.299*r + 0.587*g + 0.114*b

In [ ]:
def channel_summary(image_float: np.ndarray) -> tuple[np.ndarray, int]:
    means = image_float.mean(axis=(0, 1))
    brightest = np.argmax(means)
    return means, brightest

In [ ]:
def convolve2d_matmul(image_gray: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    h, w = image_gray.shape
    kh, kw = kernel.shape

    out_h = h - kh + 1
    out_w = w - kw + 1

    output = np.zeros((out_h, out_w), dtype=np.float32)

    k_flat = kernel.flatten()

    for i in range(out_h):
        for j in range(out_w):
            patch = image_gray[i:i+kh, j:j+kw]
            p_flat = patch.flatten()
            output[i, j] = p_flat @ k_flat

    return output

In [ ]:
def flatten_image(image: np.ndarray) -> np.ndarray:
    return image.flatten()

In [ ]:
def extract_features(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    cropped = center_crop(image, crop_size=48)
    image_float = normalize_01(cropped)
    gray = rgb_to_gray(image_float)

    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1))

    filtered = convolve2d_matmul(gray, kernel)

    edge_mean = filtered.mean()
    edge_std = filtered.std()

    row_std_profile = np.apply_along_axis(np.std, 1, gray)
    row_std_mean = row_std_profile.mean()

    features = np.concatenate([
        channel_means,
        channel_stds,
        np.array([brightest_channel]),
        np.array([edge_mean, edge_std]),
        np.array([row_std_mean])
    ])

    return features.astype(np.float32)

In [ ]:
def build_feature_matrix(paths: list[Path], kernel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    X = []
    y = []

    for path in paths:
        img = load_image_np(path)
        feat = extract_features(img, kernel)

        X.append(feat)
        y.append(LABEL_TO_INDEX[label_from_path(path)])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)

    return x, y